# Engenharia de Features — P3 Preços quebrados no catálogo

**Objetivo:** transformar a Silver de produtos em uma tabela numérica para o modelo (LightGBM) que estima o preço esperado de cada produto.

**Entrada:** Silver `ecommerce_produtos` e `ecommerce_categorias`.
**Saída:** tabela de features, uma linha por observação (micro-lote × SKU).

**Decisões herdadas da EDA (notebook 07):**
- O tamanho é extraído do `nome_produto`, e não do código do SKU.
- O preço é o nível do SKU mais ruído independente: não há feature de data nem de preço anterior.
- Marca quase não afeta o preço: entra em teste, com e sem.
- Margem fica fora da entrega (decisão do Geovany, 09/11/2026).
- Rótulo fraco (regras 02 e 07) e rótulo de apoio (preço ≤ 0 ou > 10× a mediana do SKU) servem só para avaliar.

In [0]:
# ------------------------------------------------------------
# Carga dos dados
# Lê a Silver de produtos e de categorias, as duas entradas
# deste notebook, e confere quantas linhas cada uma tem.
# ------------------------------------------------------------
import os
from dotenv import load_dotenv
from pyspark.sql import functions as F
from pyspark.sql import Window  # usado nas funções de janela (ranking, contagem por grupo)

# Lê o arquivo .env (mesma pasta do notebook), onde ficam as credenciais do ADLS.
load_dotenv(".env")

# Endereço do lake: conta de armazenamento e pasta raiz da Squad 1.
STORAGE_ACCOUNT = "internshipdatalake"
lakehouse_base = f"abfss://squad1@{STORAGE_ACCOUNT}.dfs.core.windows.net"

# Credenciais de acesso ao ADLS (autenticação OAuth com um aplicativo do Azure).
# No Serverless o spark.conf.set de credenciais é bloqueado,
# por isso elas vão em cada leitura e em cada gravação, pelo .options(**adls_options).
adls_options = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": os.getenv("ADLS_CLIENT_ID"),
    "fs.azure.account.oauth2.client.secret": os.getenv("ADLS_CLIENT_SECRET"),
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{os.getenv('ADLS_TENANT_ID')}/oauth2/token",
}

def ler_delta_spark(caminho):
    """Lê uma tabela Delta do ADLS.

    caminho: endereço abfss:// da tabela (ex.: .../silver/ecommerce_produtos).
    Retorna um DataFrame do Spark.
    """
    return spark.read.format("delta").options(**adls_options).load(caminho)

# Tabelas de entrada (camada Silver)
silver_produtos_path = f"{lakehouse_base}/silver/ecommerce_produtos"
silver_categorias_path = f"{lakehouse_base}/silver/ecommerce_categorias"

df_prod = ler_delta_spark(silver_produtos_path)   # produtos: 1 linha por (micro-lote, SKU)
df_cat = ler_delta_spark(silver_categorias_path)  # categorias: a hierarquia se repete a cada micro-lote

# Conferência: os totais têm que bater com os da EDA (notebook 07).
print("Produtos:", df_prod.count(), "linhas")
print("Categorias:", df_cat.count(), "linhas")

In [0]:
# ------------------------------------------------------------
# Passo 2: tabela com uma linha por SKU e tamanho do produto
# Os atributos do produto são fixos por SKU; só o preço muda de
# uma observação para outra. Por isso montamos uma linha por SKU
# e extraímos o tamanho a partir do nome do produto.
# ------------------------------------------------------------

# 2.1 Uma linha por SKU com os atributos fixos.
# dropDuplicates(["sku"]) mantém só uma linha de cada SKU (são 292.000 linhas para 3.805 SKUs).
sku_base = (
    df_prod
    .select("sku", "nome_produto", "id_categoria", "id_categoria_pai", "unidade_medida", "nome_marca")
    .dropDuplicates(["sku"])
)
print("SKUs:", sku_base.count())

# Conferência: algum SKU tem mais de um nome ou mais de uma unidade?
# Se tivesse, a linha escolhida pelo dropDuplicates seria arbitrária. O esperado é 0.
inconsistentes = (
    df_prod.groupBy("sku")
    .agg(
        F.countDistinct("nome_produto").alias("nomes"),
        F.countDistinct("unidade_medida").alias("unidades"),
    )
    .filter((F.col("nomes") > 1) | (F.col("unidades") > 1))
    .count()
)
print("SKUs com mais de um nome ou unidade:", inconsistentes)

# 2.2 Extrair tamanho e unidade do nome (ex.: "... 0.35L - Amstel").
# O tamanho vem do nome do produto e não do código do SKU, porque o código
# é ambíguo (15l = 1,5 L; 10kg = 10 kg) e 4 SKUs vêm truncados (achado da EDA).
# Como ler o padrão:
#   (\d+(?:\.\d+)?)  -> um número, com parte decimal opcional (35, 0.35, 1.5)
#   (kg|L|un)        -> a unidade: kg, L ou un
#   \s-\s            -> o " - " que separa o tamanho da marca
padrao = r"(\d+(?:\.\d+)?)(kg|L|un)\s-\s"
numero = F.regexp_extract("nome_produto", padrao, 1)   # grupo 1: o número
unidade = F.regexp_extract("nome_produto", padrao, 2)  # grupo 2: a unidade

# F.when(... != "") evita converter texto vazio em número: no modo ANSI do Spark
# isso daria erro. Se o padrão não for encontrado, o resultado fica nulo.
# log_tamanho é o logaritmo natural do tamanho: aproxima tamanhos muito diferentes
# (30 un x 400 un) e ajuda o modelo a enxergar proporções.
sku_base = (
    sku_base
    .withColumn("tamanho", F.when(numero != "", numero.cast("double")))
    .withColumn("unidade_no_nome", F.when(unidade != "", unidade))
    .withColumn("log_tamanho", F.log("tamanho"))
)

# 2.3 Conferência da extração (o esperado é 0 nos dois primeiros itens).
print("\n=== Conferência da extração ===")
print("SKUs sem tamanho:", sku_base.filter(F.col("tamanho").isNull()).count())
print("SKUs em que a unidade do nome difere de unidade_medida:",
      sku_base.filter(F.col("unidade_no_nome") != F.col("unidade_medida")).count())
sku_base.agg(
    F.min("tamanho").alias("menor_tamanho"),
    F.max("tamanho").alias("maior_tamanho"),
).show()

# Amostra para conferir à vista o tamanho extraído.
print("=== Amostra ===")
sku_base.select("nome_produto", "tamanho", "log_tamanho", "unidade_medida") \
    .orderBy("sku").show(8, truncate=False)

In [0]:
# ------------------------------------------------------------
# Passo 3: variante do produto
# O nome do produto tem o formato: <subcategoria> <variante> <tamanho> - <marca>
# (ex.: "Água Mineral Natural 1.5L - Bonafont").
# A variante é o que sobra depois de tirar a subcategoria e o tamanho.
# Ela diferencia produtos da mesma subcategoria e do mesmo tamanho.
# ------------------------------------------------------------
# 3.1 Nome da subcategoria de cada SKU.
# A tabela de categorias repete cada categoria 365 vezes (uma por micro-lote).
# Por isso o dropDuplicates: sem ele, o join multiplicaria cada SKU por 365.
# O join "left" mantém todos os SKUs, mesmo que algum não encontre subcategoria.
subcategorias = (
    df_cat.filter(F.col("tipo_categoria") == "Subcategoria")
    .select("id_categoria", F.col("nome_categoria").alias("subcategoria"))
    .dropDuplicates(["id_categoria"])
)
sku_base = sku_base.join(subcategorias, "id_categoria", "left")

# 3.2 Texto antes do tamanho = subcategoria + variante.
# Como ler o padrão:
#   ^(.*?)   -> captura tudo desde o início do nome, parando o mais cedo possível
#   \s*      -> espaços opcionais antes do tamanho
#   \d+(?:\.\d+)?(?:kg|L|un)\s-\s  -> o tamanho com a unidade e o " - " (o mesmo padrão do Passo 2)
padrao_antes = r"^(.*?)\s*\d+(?:\.\d+)?(?:kg|L|un)\s-\s"
sku_base = (
    sku_base
    .withColumn("antes_tamanho", F.regexp_extract("nome_produto", padrao_antes, 1))
    # Confirma que esse texto começa com o nome da subcategoria.
    .withColumn("comeca_com_subcat", F.col("antes_tamanho").startswith(F.col("subcategoria")))
    # Variante = o texto sem o prefixo da subcategoria
    # (substring corta os primeiros "tamanho da subcategoria" caracteres; trim tira espaços).
    .withColumn(
        "variante",
        F.when(
            F.col("comeca_com_subcat"),
            F.expr("trim(substring(antes_tamanho, length(subcategoria) + 1))"),
        ),
    )
    # Quando não sobra nada, usamos "sem_variante" em vez de texto vazio,
    # para o modelo tratar esses produtos como um grupo próprio.
    .withColumn(
        "variante",
        F.when(F.col("variante") == "", F.lit("sem_variante")).otherwise(F.col("variante")),
    )
)

# 3.3 Conferência
print("=== Conferência da variante ===")
print("SKUs sem nome de subcategoria:", sku_base.filter(F.col("subcategoria").isNull()).count())
print("SKUs cujo nome NÃO começa com a subcategoria:", sku_base.filter(~F.col("comeca_com_subcat")).count())
print("SKUs sem variante:", sku_base.filter(F.col("variante") == "sem_variante").count())
print("Variantes distintas:", sku_base.select("variante").distinct().count())

# Amostra e ranking para conferir à vista.
print("\n=== Amostra ===")
sku_base.select("nome_produto", "subcategoria", "variante").orderBy("sku").show(8, truncate=False)

print("=== 12 variantes mais comuns ===")
sku_base.groupBy("variante").count().orderBy(F.desc("count")).show(12, truncate=False)

# Resumo: quantas variantes cada subcategoria tem (mínimo, média e máximo).
print("=== Quantas variantes cada subcategoria tem (resumo) ===")
(
    sku_base.groupBy("id_categoria")
    .agg(F.countDistinct("variante").alias("variantes"))
    .agg(
        F.min("variantes").alias("minimo"),
        F.round(F.avg("variantes"), 1).alias("media"),
        F.max("variantes").alias("maximo"),
    )
    .show()
)

In [0]:
# ------------------------------------------------------------
# Passo 4: features de marca
# Na EDA, a marca quase não mexeu no preço (±2%, menos que o acaso).
# Mesmo assim ela entra, para o modelo testar com e sem marca.
# Para o teste "sem marca", basta tirar marca_cod, marca_n_skus,
# flag_generico e flag_marca_propria da lista FEATURES (Passo 6).
# ------------------------------------------------------------

# 4.1 Flags de marca branca (1 = sim, 0 = não).
# "Genérico" e "Marca Própria" não são marcas de verdade: agrupam produtos
# de várias origens, e o modelo pode tratá-los de forma diferente.
sku_base = (
    sku_base
    .withColumn("flag_generico", (F.col("nome_marca") == "Genérico").cast("int"))
    .withColumn("flag_marca_propria", (F.col("nome_marca") == "Marca Própria").cast("int"))
)

# 4.2 Quantos SKUs cada marca tem.
# Marcas com poucos SKUs dão pouca informação ao modelo; esta coluna deixa isso explícito.
# (Window.partitionBy("nome_marca") calcula a contagem dentro de cada marca.)
sku_base = sku_base.withColumn(
    "marca_n_skus", F.count("*").over(Window.partitionBy("nome_marca"))
)

# 4.3 Código numérico da marca, em ordem alfabética (1 a 239).
# A ordem não tem significado de preço: o LightGBM deve tratar marca_cod como categoria.
# O aviso "No Partition Defined for Window operation" é só de desempenho e pode ser ignorado:
# são apenas 3.805 linhas.
sku_base = sku_base.withColumn(
    "marca_cod", F.dense_rank().over(Window.orderBy("nome_marca"))
)

# 4.4 Conferência
print("=== Conferência da marca ===")
print("SKUs Genérico:", sku_base.agg(F.sum("flag_generico")).collect()[0][0])
print("SKUs Marca Própria:", sku_base.agg(F.sum("flag_marca_propria")).collect()[0][0])
print("Marcas distintas:", sku_base.select("nome_marca").distinct().count())
print("Maior marca_cod:", sku_base.agg(F.max("marca_cod")).collect()[0][0])

# Uma linha por marca, para contar quantas marcas têm poucos SKUs.
marcas = sku_base.select("nome_marca", "marca_n_skus", "marca_cod").distinct()
print("Marcas com apenas 1 SKU:", marcas.filter(F.col("marca_n_skus") == 1).count())
print("Marcas com 5 SKUs ou menos:", marcas.filter(F.col("marca_n_skus") <= 5).count())

print("\n=== Amostra de marcas ===")
marcas.orderBy("marca_cod").show(8, truncate=False)

In [0]:
# ------------------------------------------------------------
# Passo 5: tamanho relativo, posição do tamanho e códigos numéricos
# O mesmo tamanho pesa de formas diferentes: "30 un" é um pacote grande
# em uma subcategoria e pequeno em outra. Estas colunas descrevem o
# tamanho em relação aos demais produtos da mesma subcategoria.
# ------------------------------------------------------------

# 5.1 Tamanho relativo = tamanho dividido pelo tamanho típico (mediana) da subcategoria.
# Agrupa por subcategoria E unidade, porque não faz sentido comparar kg com L.
# Valor 1 = tamanho típico; 2 = o dobro do típico; 0,5 = metade.
med_tamanho = sku_base.groupBy("id_categoria", "unidade_medida").agg(
    F.percentile_approx("tamanho", 0.5).alias("tamanho_mediano_subcat")
)
sku_base = (
    sku_base
    .join(med_tamanho, ["id_categoria", "unidade_medida"], "left")
    .withColumn("tamanho_relativo", F.col("tamanho") / F.col("tamanho_mediano_subcat"))
)

# 5.2 Posição do tamanho entre os tamanhos DISTINTOS da subcategoria (0 = menor, 1 = maior).
# Usa dense_rank sobre os tamanhos diferentes, e não sobre os SKUs: assim a posição
# não muda conforme o número de SKUs de cada tamanho. Se a subcategoria só tem
# um tamanho, não há o que ordenar e usamos 0,5.
# _posicao e _n_tamanhos são colunas auxiliares, apagadas no final.
janela_ordem = Window.partitionBy("id_categoria", "unidade_medida").orderBy("tamanho")
janela_grupo = Window.partitionBy("id_categoria", "unidade_medida")
sku_base = (
    sku_base
    .withColumn("_posicao", F.dense_rank().over(janela_ordem))
    .withColumn("_n_tamanhos", F.max("_posicao").over(janela_grupo))
    .withColumn(
        "tamanho_rank_pct",
        F.when(F.col("_n_tamanhos") > 1, (F.col("_posicao") - 1) / (F.col("_n_tamanhos") - 1))
         .otherwise(F.lit(0.5)),
    )
    .drop("_posicao", "_n_tamanhos")
)

# 5.3 Códigos numéricos para o modelo (o LightGBM não lê texto).
# A variante é codificada junto com a subcategoria, porque o mesmo nome
# (ex.: "Tradicional") aparece em várias subcategorias e não é o mesmo produto.
# Os códigos seguem a ordem alfabética e não têm significado de preço:
# devem ser tratados como categorias.
sku_base = (
    sku_base
    .withColumn("chave_variante", F.concat_ws("|", F.col("id_categoria").cast("string"), F.col("variante")))
    .withColumn("variante_cod", F.dense_rank().over(Window.orderBy("chave_variante")))
    .withColumn("unidade_cod", F.dense_rank().over(Window.orderBy("unidade_medida")))
)

# 5.4 Conferência: nenhuma coluna nova pode ter nulos.
print("=== Conferência ===")
colunas_novas = ["tamanho_relativo", "tamanho_rank_pct", "variante_cod", "unidade_cod"]
for c in colunas_novas:
    print(f"{c}: {sku_base.filter(F.col(c).isNull()).count()} nulos")

print("Pares subcategoria + variante distintos:", sku_base.select("chave_variante").distinct().count())
print("Maior variante_cod:", sku_base.agg(F.max("variante_cod")).collect()[0][0])

# A posição do tamanho tem que ir de 0 a 1.
sku_base.agg(
    F.min("tamanho_rank_pct").alias("rank_minimo"),
    F.max("tamanho_rank_pct").alias("rank_maximo"),
).show()

print("=== Códigos de unidade ===")
sku_base.select("unidade_medida", "unidade_cod").distinct().orderBy("unidade_cod").show()

# Exemplo para ver as três colunas de tamanho lado a lado.
print("=== Exemplo: tamanhos em Fraldas ===")
(
    sku_base.filter(F.col("subcategoria") == "Fraldas")
    .select("tamanho", "tamanho_mediano_subcat", "tamanho_relativo", "tamanho_rank_pct")
    .distinct()
    .orderBy("tamanho")
    .show()
)

In [0]:
# ------------------------------------------------------------
# Passo 6: lista oficial de features e tabela final por SKU
# Esta célula define o que o modelo (notebook 09) recebe como entrada.
# ------------------------------------------------------------

# Features categóricas (5): códigos sem ordem, que identificam um grupo.
# O LightGBM deve ser avisado de que estas colunas são categorias.
FEATURES_CATEGORICAS = ["id_categoria", "id_categoria_pai", "unidade_cod", "variante_cod", "marca_cod"]
#   id_categoria = subcategoria | id_categoria_pai = categoria raiz | unidade_cod = L, kg ou un
#   variante_cod = par subcategoria + variante   | marca_cod = marca

# Features numéricas (7).
FEATURES_NUMERICAS = [
    "tamanho", "log_tamanho", "tamanho_relativo", "tamanho_rank_pct",
    "marca_n_skus", "flag_generico", "flag_marca_propria",
]

# Lista oficial: 12 features.
# Para o teste "sem marca", tire daqui: marca_cod, marca_n_skus, flag_generico e flag_marca_propria.
FEATURES = FEATURES_CATEGORICAS + FEATURES_NUMERICAS

# Colunas de texto, só para leitura humana (NÃO são features).
COLUNAS_TEXTO = ["subcategoria", "variante", "nome_marca", "nome_produto"]

# Tabela final, uma linha por SKU. Os ids viram inteiros (int) porque o LightGBM
# espera códigos inteiros nas colunas categóricas.
# Nenhuma feature usa o preço: o preço só entra depois, no Passo 7, como alvo e para os rótulos.
features_sku = (
    sku_base
    .withColumn("id_categoria", F.col("id_categoria").cast("int"))
    .withColumn("id_categoria_pai", F.col("id_categoria_pai").cast("int"))
    .select("sku", *COLUNAS_TEXTO, *FEATURES)
)

# Conferência
print("Linhas:", features_sku.count(), "| SKUs distintos:", features_sku.select("sku").distinct().count())
print("Quantidade de features:", len(FEATURES))

print("\n=== Tipos das colunas ===")
features_sku.printSchema()

# Nenhuma feature pode ter nulo.
nulos = features_sku.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in FEATURES]).collect()[0]
print("Nulos por feature:", {c: nulos[c] for c in FEATURES})

print("\n=== Resumo das features numéricas ===")
features_sku.select(FEATURES_NUMERICAS).summary("min", "50%", "max").show()

print("=== Exemplo de 2 SKUs ===")
features_sku.orderBy("sku").show(2, truncate=False, vertical=True)

In [0]:
# ------------------------------------------------------------
# Passo 7: observações com features e rótulos
# Aqui o preço entra pela primeira vez. Juntamos as features de cada SKU
# a cada observação de preço (micro-lote x SKU) e criamos os rótulos.
# Os rótulos servem só para treinar e avaliar. Nunca são features.
# ------------------------------------------------------------

# 7.1 Nível (mediana) de cada SKU, só com preços positivos.
# Usamos a mediana porque os erros gigantes não a puxam, ao contrário da média.
# É usada SÓ para montar rótulos: se virasse feature, o modelo "colaria" o preço (vazamento).
mediana_sku = (
    df_prod.filter(F.col("preco_lista") > 0)
    .groupBy("sku")
    .agg(F.percentile_approx(F.col("preco_lista").cast("double"), 0.5).alias("mediana_sku"))
)

# 7.2 Observações com features e rótulos.
# Uma linha por (micro-lote, SKU): 292.000 linhas.
# O join "left" com a mediana mantém SKUs que, por algum motivo, não tenham preço positivo.
obs = (
    df_prod
    .select(
        "sku",
        "bronze_source_file",                                 # identifica o micro-lote
        F.col("preco_lista").cast("double").alias("preco"),   # alvo do modelo
        "regra_02_preco_positivo",
        "regra_07_preco_dentro_limite",
    )
    .join(features_sku.select("sku", *FEATURES), "sku")
    .join(mediana_sku, "sku", "left")
    # Quantas vezes o preço é maior que o nível normal do SKU (1 = normal; 10 = dez vezes maior).
    .withColumn("razao_mediana", F.col("preco") / F.col("mediana_sku"))
    # Rótulo fraco: o que as regras de qualidade da Etapa 1 já marcam
    # (regra 02 = preço positivo; regra 07 = preço dentro do limite). 1 = marcado.
    .withColumn(
        "rotulo_fraco",
        ((F.col("regra_02_preco_positivo") == False) | (F.col("regra_07_preco_dentro_limite") == False)).cast("int"),
    )
    # Rótulo de apoio (aprovado pelo Geovany, só para avaliar): preço zero, negativo
    # ou acima de 10x a mediana do SKU. Enxerga erros que as regras deixam passar.
    # coalesce(..., False) trata o caso em que a razão é nula.
    .withColumn(
        "rotulo_apoio",
        F.coalesce((F.col("preco") <= 0) | (F.col("razao_mediana") > 10), F.lit(False)).cast("int"),
    )
    # Linhas que entram no treino: preço positivo e até 10x a mediana do SKU,
    # ou seja, sem os erros óbvios. É o complemento do rótulo de apoio.
    .withColumn(
        "usar_no_treino",
        F.coalesce((F.col("preco") > 0) & (F.col("razao_mediana") <= 10), F.lit(False)).cast("int"),
    )
)

# 7.3 Conferência
print("Linhas:", obs.count())
print("Linhas sem razao_mediana:", obs.filter(F.col("razao_mediana").isNull()).count())

print("\n=== Totais ===")
obs.agg(
    F.sum("rotulo_fraco").alias("rotulo_fraco"),
    F.sum("rotulo_apoio").alias("rotulo_apoio"),
    F.sum("usar_no_treino").alias("usar_no_treino"),
).show()

# Cruzamento dos dois rótulos: mostra quantos erros só o rótulo de apoio enxerga.
print("=== Rótulo fraco x rótulo de apoio ===")
obs.groupBy("rotulo_fraco", "rotulo_apoio").count().orderBy("rotulo_fraco", "rotulo_apoio").show()

# Trava de segurança: nenhuma linha de erro pode entrar no treino.
print("Linhas que entram no treino E são rótulo de apoio (deve ser 0):",
      obs.filter((F.col("usar_no_treino") == 1) & (F.col("rotulo_apoio") == 1)).count())

In [0]:
# ------------------------------------------------------------
# Passo 8: separação treino e teste, por SKU
# Dividimos os SKUs em 5 grupos (folds). O fold 0 é o TESTE; os folds 1 a 4
# são o TREINO e também servem para validação cruzada.
# A divisão é por SKU, e não por linha: como as features quase identificam
# o SKU, se linhas do mesmo SKU ficassem dos dois lados, o modelo
# decoraria o preço de cada SKU e a avaliação seria falsamente boa.
# ------------------------------------------------------------
N_FOLDS = 5

# Rodízio dentro de cada subcategoria: os SKUs de cada subcategoria são
# colocados em ordem (pelo hash do código do SKU, uma ordem embaralhada, mas
# sempre a mesma, o que torna a divisão reproduzível) e distribuídos
# nos folds 0, 1, 2, 3, 4, 0, 1... Assim cada subcategoria fica espalhada pelos
# folds e nenhuma fica inteira no teste.
# deslocamento: o ponto de partida do rodízio muda de subcategoria para subcategoria,
# para o fold 0 não ficar sempre com os SKUs que sobram.
# Subcategoria com um único SKU vai direto para o fold 1 (treino), para o teste
# nunca ter uma subcategoria que o treino não conhece.
janela_subcat = Window.partitionBy("id_categoria").orderBy("hash_sku", "sku")
sku_folds = (
    features_sku.select("sku", "id_categoria")
    .withColumn("hash_sku", F.hash("sku"))
    .withColumn("posicao", F.row_number().over(janela_subcat))
    .withColumn("n_skus_subcat", F.count("*").over(Window.partitionBy("id_categoria")))
    .withColumn("deslocamento", F.pmod(F.hash("id_categoria"), F.lit(N_FOLDS)))
    .withColumn(
        "fold",
        F.when(F.col("n_skus_subcat") == 1, F.lit(1))
         .otherwise(F.pmod(F.col("posicao") - 1 + F.col("deslocamento"), F.lit(N_FOLDS))),
    )
    .select("sku", "fold")
)

# Cola o fold em cada observação. O drop() no início permite rodar a célula de novo
# sem duplicar as colunas fold e conjunto.
obs = (
    obs.drop("fold", "conjunto")
    .join(sku_folds, "sku")
    .withColumn("conjunto", F.when(F.col("fold") == 0, "teste").otherwise("treino"))
)

# 8.1 Tamanho de cada fold (os folds devem ter tamanhos parecidos).
print("=== Por fold ===")
obs.groupBy("fold").agg(
    F.countDistinct("sku").alias("skus"),
    F.count("*").alias("linhas"),
    F.sum("usar_no_treino").alias("usar_no_treino"),
    F.sum("rotulo_fraco").alias("rotulo_fraco"),
    F.sum("rotulo_apoio").alias("rotulo_apoio"),
).orderBy("fold").show()

# 8.2 Algum SKU caiu em mais de um fold? (deve ser 0)
repetidos = (
    obs.groupBy("sku").agg(F.countDistinct("fold").alias("n_folds"))
    .filter(F.col("n_folds") > 1).count()
)
print("SKUs em mais de um fold:", repetidos)

# 8.3 O teste tem subcategorias, variantes e marcas que o treino nunca viu?
# (um modelo não consegue prever bem um grupo que nunca viu no treino)
treino = obs.filter(F.col("conjunto") == "treino")
teste = obs.filter(F.col("conjunto") == "teste")
print("\n=== Cobertura do teste ===")
for coluna in ["id_categoria", "variante_cod", "marca_cod"]:
    no_teste = teste.select(coluna).distinct()
    no_treino = treino.select(coluna).distinct()
    sem_treino = no_teste.join(no_treino, coluna, "left_anti").count()
    print(f"{coluna}: {no_teste.count()} valores no teste, {sem_treino} não aparecem no treino")

In [0]:
# ------------------------------------------------------------
# Passo 9: gravação das tabelas no lake
# Grava as duas tabelas Delta que o notebook de modelagem (09) vai ler.
# ATENÇÃO: esta é a única célula que ESCREVE no lake. Ela sobrescreve as
# duas tabelas a cada execução completa (os dados anteriores ficam no
# histórico do Delta).
# ------------------------------------------------------------
ml_base = f"{lakehouse_base}/ml"                                   # pasta de ML da Squad 1
caminho_features_sku = f"{ml_base}/features_precos_sku"            # uma linha por SKU
caminho_features_obs = f"{ml_base}/features_precos_observacoes"    # uma linha por observação

# Colunas da tabela de observações (22), por grupo:
#   identificação:   sku, bronze_source_file (micro-lote)
#   alvo:            preco
#   features (12):   a lista FEATURES do Passo 6
#   só avaliação:    mediana_sku, razao_mediana (derivam do preço: NÃO são features)
#   rótulos:         rotulo_fraco, rotulo_apoio, usar_no_treino
#   divisão:         fold (0 a 4) e conjunto ("teste" para o fold 0, "treino" para os outros)
obs_final = obs.select(
    "sku", "bronze_source_file", "preco",
    *FEATURES,
    "mediana_sku", "razao_mediana",   # só para avaliar, NÃO são features
    "rotulo_fraco", "rotulo_apoio", "usar_no_treino",
    "fold", "conjunto",
)

# mode("overwrite") troca o conteúdo da tabela; overwriteSchema permite que o
# conjunto de colunas mude entre versões. As credenciais vão pelo .options().
(
    features_sku.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").options(**adls_options)
    .save(caminho_features_sku)
)
(
    obs_final.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").options(**adls_options)
    .save(caminho_features_obs)
)

# Conferência: lê de volta do lake o que foi gravado (e não do que está na memória),
# para provar que a gravação deu certo.
sku_lido = ler_delta_spark(caminho_features_sku)
obs_lido = ler_delta_spark(caminho_features_obs)

print("features_precos_sku:", sku_lido.count(), "linhas,", len(sku_lido.columns), "colunas")
print("features_precos_observacoes:", obs_lido.count(), "linhas,", len(obs_lido.columns), "colunas")

print("\n=== Totais lidos do lake ===")
obs_lido.agg(
    F.sum("rotulo_fraco").alias("rotulo_fraco"),
    F.sum("rotulo_apoio").alias("rotulo_apoio"),
    F.sum("usar_no_treino").alias("usar_no_treino"),
).show()

print("=== Linhas por conjunto ===")
obs_lido.groupBy("conjunto").agg(
    F.countDistinct("sku").alias("skus"),
    F.count("*").alias("linhas"),
).show()

# Conclusões da Engenharia de Features

## 1. O que foi gerado
Duas tabelas Delta no lake, em `squad1/ml/`:
- `features_precos_sku`: uma linha por SKU (3.805), com as features e os textos para leitura.
- `features_precos_observacoes`: uma linha por observação micro-lote × SKU (292.000), com o preço, as features, os rótulos e o `fold`. É a tabela que o modelo vai ler.

## 2. Features (12)
- **Categóricas (códigos, sem ordem):** `id_categoria` (subcategoria), `id_categoria_pai` (categoria raiz), `unidade_cod`, `variante_cod` (par subcategoria + variante) e `marca_cod`.
- **Numéricas:** `tamanho`, `log_tamanho`, `tamanho_relativo` (tamanho ÷ mediana da subcategoria), `tamanho_rank_pct` (posição entre os tamanhos da subcategoria, de 0 a 1), `marca_n_skus`, `flag_generico` e `flag_marca_propria`.
- O tamanho vem do `nome_produto`. O código do SKU é ambíguo e 4 SKUs vêm truncados.
- Nenhuma feature usa o preço. `mediana_sku` e `razao_mediana` existem só para montar rótulos e avaliar.
- A marca entra para ser testada com e sem (basta tirar `marca_cod`, `marca_n_skus` e as duas flags da lista `FEATURES`).

## 3. Rótulos (só para treinar e avaliar)
- `rotulo_fraco`: regra 02 ou 07 falhou (4.443 linhas).
- `rotulo_apoio`: preço ≤ 0 ou acima de 10× a mediana do SKU (6.551 linhas), aprovado pelo Geovany.
- O rótulo fraco enxerga 67,8% do que o de apoio enxerga; 2.108 erros só aparecem no de apoio.
- `usar_no_treino`: preço positivo e até 10× a mediana do SKU (285.449 linhas).

## 4. Separação treino/teste
- 5 folds por SKU, em rodízio dentro de cada subcategoria: todas as linhas de um SKU ficam do mesmo lado, e nenhuma subcategoria fica inteira no teste.
- Teste = `fold` 0 (765 SKUs, 58.546 linhas). Treino = folds 1 a 4 (3.040 SKUs, 233.454 linhas), que também servem para validação cruzada.
- O teste cobre 113 das 123 subcategorias. Todas as subcategorias e variantes do teste existem no treino; 2 marcas aparecem só no teste.

## 5. Próximo passo
Modelagem com LightGBM: treinar com `usar_no_treino = 1`, comparar com a escadinha de baselines da EDA (MAE de R$ 9,23 para subcategoria + tamanho e R$ 9,04 de piso), gerar o resíduo robusto (MAD) e avaliar o alerta contra os dois rótulos.

## 6. Como usar as tabelas no notebook 09 (plano)

```python
obs = ler_delta_spark(f"{lakehouse_base}/ml/features_precos_observacoes")
treino = obs.filter((F.col("conjunto") == "treino") & (F.col("usar_no_treino") == 1))
teste = obs.filter(F.col("conjunto") == "teste")
```

- **Treinar:** só com as linhas de `treino` que tenham `usar_no_treino = 1` (233.454 linhas menos os erros óbvios).
- **Avaliar o preço esperado (MAE):** nas linhas de `teste` com `usar_no_treino = 1`, para comparar com a escadinha de baselines da EDA.
- **Avaliar o alerta (precision, recall, taxa de alerta):** em **todas** as linhas de `teste`, contra `rotulo_fraco` e `rotulo_apoio`.
- **Entradas do modelo:** as 12 colunas da lista `FEATURES`. As cinco primeiras (ids e códigos) devem ser declaradas como categóricas no LightGBM.
- **Fora do modelo:** `mediana_sku` e `razao_mediana` (derivam do preço), os rótulos, `fold`, `conjunto` e as colunas de identificação.